# Install Dependencies

In [1]:
!pip install -q datasets scikit-learn pandas numpy

# Data Collection

In [2]:
%%writefile data_collection.py
import pandas as pd
import os

REQUIRED_COLUMNS = ["unique_id", "claim", "company"]

def load_from_huggingface() -> pd.DataFrame:
    from datasets import load_dataset
    ds = load_dataset("geoka/EmeraldData", split="train")
    return _clean(ds.to_pandas())

def load_from_csv(path: str = "emerald_sample.csv") -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Could not find {path}.")
    return _clean(pd.read_csv(path))

def _clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["claim"] = df["claim"].astype(str).str.strip()
    df = df[df["claim"].str.len() > 0].reset_index(drop=True)
    if "gold_label" in df.columns:
        df["gold_label"] = df["gold_label"].astype(str).str.strip().str.lower().str.replace(" ", "_")
    if "company" in df.columns:
        df["company"] = df["company"].astype(str).str.strip()
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Dataset is missing required columns: {missing}")
    return df

def load_dataset_auto(prefer_full: bool = False) -> pd.DataFrame:
    if prefer_full:
        try:
            return load_from_huggingface()
        except Exception as e:
            print(f"[data_collection] Falling back to local CSV — reason: {e}")
    return load_from_csv()

Writing data_collection.py


# Claim extraction

In [3]:
%%writefile claim_extraction.py
import re

ENV_KEYWORDS = [
    "sustain", "green", "eco", "climate", "carbon", "emission", "renewable",
    "recycl", "compost", "biodegrad", "plastic", "waste", "environment",
    "net zero", "net-zero", "clean energy", "circular", "organic", "natural",
    "footprint", "conservation", "responsibly sourced", "fair trade",
    "regenerative", "deforestation", "clean", "solar", "wind power",
]
YEAR_PATTERN = re.compile(r"\b(19|20)\d{2}\b")

def split_into_claims(ad_text: str) -> list[str]:
    sentences = re.split(r"(?<=[.!?])\s+", ad_text.strip())
    return [s.strip() for s in sentences if len(s.strip()) > 8]

def is_environmental_claim(claim: str) -> bool:
    lowered = claim.lower()
    return any(kw in lowered for kw in ENV_KEYWORDS)

def extract_metadata(claim: str) -> dict:
    return {
        "word_count": len(claim.split()),
        "has_year": bool(YEAR_PATTERN.search(claim)),
        "years_mentioned": YEAR_PATTERN.findall(claim),
        "is_environmental_claim": is_environmental_claim(claim),
    }

Writing claim_extraction.py


# NLP Classifier

In [4]:
%%writefile nlp_classifier.py
import re
from dataclasses import dataclass, field

VAGUE_TERMS = [
    "eco-friendly", "green", "natural", "sustainable", "responsible",
    "climate-friendly", "clean", "planet-friendly", "conscious",
    "better for the environment", "kind to the earth", "eco-conscious",
    "earth-friendly", "environmentally friendly", "resource positive",
    "positive hydration", "lasting impact", "sustainable future",
    "sustainable journey", "vibrant future", "significant impact",
]
UNSUPPORTED_SUPERLATIVES = [
    "truly", "leading", "leader", "pioneer", "first", "best", "revolutionary",
    "groundbreaking", "innovative", "change-your-life", "solves", "solving",
    "eliminate", "eliminating", "zero", "100%", "guarantee", "guarantees",
    "proudly", "new industry benchmark", "global leader", "exemplif",
]
OVERGENERALIZATION_SCOPE_WORDS = [
    "all ", "every ", "entire", "across its operations", "all customers",
    "all products", "industry-wide", "across the u.s.", "widely available",
    "accessible and widely", "for all customers",
]
QUALIFIER_WORDS = [
    "aims to", "aspires", "aspiration", "working towards", "committed to",
    "plans to", "intends to", "hopes to", "on its journey", "roadmap",
    "by 2030", "by 2050", "by 2025",
]
COMPARATIVE_WORDS = [
    "more sustainable", "more eco-friendly", "better than", "compared to",
    "than older", "than conventional", "reduced by", "increase", "decrease",
]
EVIDENCE_MARKERS = [
    r"\d+(\.\d+)?%", r"\$\s?\d+", r"\d+\s?(million|billion|thousand)",
    r"\b(19|20)\d{2}\b", r"\bSBTi\b", r"science based targets", r"\bISO\s?\d+\b",
    r"\bB\s?Corp\b", r"\bLEED\b", r"\bFSC\b", r"\bEnergy Star\b", r"\bFair\s?Trade\b",
    r"\bEPA\b", r"\bFirst Movers Coalition\b",
]

def _count_hits(text_lower: str, terms) -> list:
    return [t for t in terms if t.lower() in text_lower]

def _count_regex_hits(text: str, patterns) -> list:
    hits = []
    for p in patterns:
        m = re.search(p, text, flags=re.IGNORECASE)
        if m:
            hits.append(m.group(0))
    return hits

@dataclass
class RuleBasedResult:
    claim: str
    vague_hits: list = field(default_factory=list)
    superlative_hits: list = field(default_factory=list)
    scope_hits: list = field(default_factory=list)
    qualifier_hits: list = field(default_factory=list)
    comparative_hits: list = field(default_factory=list)
    evidence_hits: list = field(default_factory=list)
    claim_type: str = "specific"
    risk_score: int = 0
    predicted_label: str = "not_greenwashing"

def classify_rule_based(claim: str) -> RuleBasedResult:
    lowered = claim.lower()
    vague_hits = _count_hits(lowered, VAGUE_TERMS)
    superlative_hits = _count_hits(lowered, UNSUPPORTED_SUPERLATIVES)
    scope_hits = _count_hits(lowered, OVERGENERALIZATION_SCOPE_WORDS)
    qualifier_hits = _count_hits(lowered, QUALIFIER_WORDS)
    comparative_hits = _count_hits(lowered, COMPARATIVE_WORDS)
    evidence_hits = _count_regex_hits(claim, EVIDENCE_MARKERS)

    if comparative_hits:
        claim_type = "comparative"
    elif qualifier_hits and not evidence_hits:
        claim_type = "qualified"
    elif vague_hits and not evidence_hits:
        claim_type = "vague"
    else:
        claim_type = "specific"

    risk_score = (2*len(vague_hits) + 2*len(superlative_hits) + 2*len(scope_hits)
                  + 1*len(qualifier_hits) - 3*len(evidence_hits))
    predicted_label = "greenwashing" if risk_score >= 2 else "not_greenwashing"

    return RuleBasedResult(claim=claim, vague_hits=vague_hits, superlative_hits=superlative_hits,
        scope_hits=scope_hits, qualifier_hits=qualifier_hits, comparative_hits=comparative_hits,
        evidence_hits=evidence_hits, claim_type=claim_type, risk_score=risk_score,
        predicted_label=predicted_label)

class MLClassifier:
    def __init__(self):
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.linear_model import LogisticRegression
        self.vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=2, stop_words="english", max_features=3000)
        self.model = LogisticRegression(max_iter=1000, class_weight="balanced")
        self.is_fitted = False

    def fit(self, claims, labels):
        X = self.vectorizer.fit_transform(claims)
        self.model.fit(X, labels)
        self.is_fitted = True
        return self

    def predict(self, claims):
        X = self.vectorizer.transform(claims)
        return self.model.predict(X).tolist()

    def predict_proba_greenwashing(self, claims):
        X = self.vectorizer.transform(claims)
        classes = list(self.model.classes_)
        idx = classes.index("greenwashing") if "greenwashing" in classes else 0
        return self.model.predict_proba(X)[:, idx].tolist()

    def top_features(self, n=15):
        import numpy as np
        feature_names = self.vectorizer.get_feature_names_out()
        classes = list(self.model.classes_)
        idx = classes.index("greenwashing") if "greenwashing" in classes else 0
        coefs = self.model.coef_[0] if len(classes)==2 else self.model.coef_[idx]
        top_idx = np.argsort(coefs)[-n:][::-1]
        return [(feature_names[i], round(float(coefs[i]),3)) for i in top_idx]

Writing nlp_classifier.py


# Evidence Analyzer

In [5]:
%%writefile evidence_analyzer.py
from dataclasses import dataclass, field
from nlp_classifier import classify_rule_based, RuleBasedResult

INDUSTRY_MAP = {
    "delta": "Aviation", "lufthansa": "Aviation", "starbucks": "Food & Beverage",
    "keurig": "Food & Beverage", "amazon": "E-commerce / Tech", "evgo": "EV Infrastructure",
    "shein": "Fast Fashion",
}

def guess_industry(company: str) -> str:
    lowered = (company or "").lower()
    for key, industry in INDUSTRY_MAP.items():
        if key in lowered:
            return industry
    return "Other / Unclassified"

@dataclass
class EvidenceReport:
    claim: str
    specificity_score: int = 0
    evidence_gap_flags: list = field(default_factory=list)
    greenwashing_type: str = "None detected"
    industry: str = "Other / Unclassified"

def analyze_evidence(claim: str, company: str = "", rb: RuleBasedResult = None) -> EvidenceReport:
    rb = rb or classify_rule_based(claim)
    flags = []
    specificity = 5 + 2*len(rb.evidence_hits) - len(rb.vague_hits) - len(rb.superlative_hits)
    specificity = max(0, min(10, specificity))

    if not rb.evidence_hits:
        flags.append("No verifiable number, date, or third-party standard cited.")
    if rb.scope_hits:
        flags.append(f"Scope word(s) {rb.scope_hits} suggest the claim may generalize beyond what's proven.")
    if rb.qualifier_hits and not rb.evidence_hits:
        flags.append(f"Framed as a future intention ({rb.qualifier_hits}) rather than a current, checkable fact.")
    if rb.superlative_hits:
        flags.append(f"Unsupported superlative language: {rb.superlative_hits}.")
    if rb.vague_hits and not rb.evidence_hits:
        flags.append(f"Vague/undefined sustainability term(s): {rb.vague_hits}.")

    if rb.vague_hits and not rb.evidence_hits and not rb.scope_hits:
        g_type = "Type 1 - Misleading / vague label"
    elif rb.qualifier_hits and rb.superlative_hits and "benchmark" in claim.lower():
        g_type = "Type 2 - Baseline presented as achievement"
    elif rb.scope_hits:
        g_type = "Type 3 - Overgeneralization"
    elif rb.superlative_hits and not rb.evidence_hits:
        g_type = "Type 4 - Unsupported claim"
    elif rb.predicted_label == "greenwashing":
        g_type = "Type 4 - Unsupported claim"
    else:
        g_type = "None detected"

    return EvidenceReport(claim=claim, specificity_score=specificity, evidence_gap_flags=flags,
        greenwashing_type=g_type, industry=guess_industry(company))

Writing evidence_analyzer.py


# Explainability Engine

In [6]:
%%writefile explainability_engine.py
from dataclasses import dataclass, field
from nlp_classifier import classify_rule_based
from evidence_analyzer import analyze_evidence

@dataclass
class Verdict:
    claim: str
    company: str
    final_label: str
    confidence: int
    claim_type: str
    specificity_score: int
    greenwashing_type: str
    industry: str
    reasons: list = field(default_factory=list)
    highlighted_terms: dict = field(default_factory=dict)
    gold_label: str = None
    ml_label: str = None
    ml_probability: float = None

def build_verdict(claim, company="", gold_label=None, ml_label=None, ml_probability=None) -> Verdict:
    rb = classify_rule_based(claim)
    ev = analyze_evidence(claim, company, rb=rb)

    raw = rb.risk_score
    confidence = int(max(5, min(95, 50 + raw*8)))
    if rb.predicted_label == "not_greenwashing":
        confidence = max(5, min(95, 100 - confidence))

    reasons = []
    if rb.evidence_hits:
        reasons.append(f"Cites checkable evidence: {', '.join(rb.evidence_hits[:3])}")
    else:
        reasons.append("No numbers, dates, or third-party certifications cited.")
    if rb.vague_hits:
        reasons.append(f"Uses vague sustainability language: {', '.join(rb.vague_hits)}")
    if rb.scope_hits:
        reasons.append(f"Broad scope wording may overstate reach: {', '.join(rb.scope_hits)}")
    if rb.superlative_hits:
        reasons.append(f"Unsupported superlatives: {', '.join(rb.superlative_hits)}")
    if rb.comparative_hits:
        reasons.append("Comparative claim — check the baseline it's compared against.")
    reasons.extend(f"{flag}" for flag in ev.evidence_gap_flags[:2])

    highlighted = {}
    for term in rb.vague_hits: highlighted[term] = "vague"
    for term in rb.superlative_hits: highlighted[term] = "superlative"
    for term in rb.scope_hits: highlighted[term] = "scope"
    for term in rb.evidence_hits: highlighted[term] = "evidence"

    return Verdict(claim=claim, company=company, final_label=rb.predicted_label, confidence=confidence,
        claim_type=rb.claim_type, specificity_score=ev.specificity_score, greenwashing_type=ev.greenwashing_type,
        industry=ev.industry, reasons=reasons, highlighted_terms=highlighted, gold_label=gold_label,
        ml_label=ml_label, ml_probability=ml_probability)

Writing explainability_engine.py


# Run Pipeline

In [7]:
%%writefile run_pipeline.py
import argparse, json
from collections import Counter
from data_collection import load_dataset_auto, load_from_csv
from claim_extraction import extract_metadata
from explainability_engine import build_verdict

def compute_metrics(df_results):
    def prf(y_true, y_pred, positive="greenwashing"):
        tp = sum(1 for t,p in zip(y_true,y_pred) if t==positive and p==positive)
        fp = sum(1 for t,p in zip(y_true,y_pred) if t!=positive and p==positive)
        fn = sum(1 for t,p in zip(y_true,y_pred) if t==positive and p!=positive)
        tn = sum(1 for t,p in zip(y_true,y_pred) if t!=positive and p!=positive)
        acc = (tp+tn)/max(1,len(y_true)); prec = tp/max(1,(tp+fp)); rec = tp/max(1,(tp+fn))
        f1 = 2*prec*rec/max(1e-9,(prec+rec))
        return {"accuracy":round(acc,3),"precision":round(prec,3),"recall":round(rec,3),"f1":round(f1,3),
                "tp":tp,"fp":fp,"fn":fn,"tn":tn}
    gold = [r["gold_label"] for r in df_results if r.get("gold_label")]
    rule_pred = [r["final_label"] for r in df_results if r.get("gold_label")]
    report = {"n_evaluated": len(gold), "rule_based (all rows)": prf(gold, rule_pred)}
    test_rows = [r for r in df_results if r.get("gold_label") and r.get("ml_label") and r.get("is_ml_test_row")]
    ml_pred = [r["ml_label"] for r in test_rows]; ml_gold = [r["gold_label"] for r in test_rows]
    if ml_pred:
        report["ml_model (held-out test split only)"] = prf(ml_gold, ml_pred)
        report["ml_test_set_size"] = len(ml_pred)
    return report

def main(use_full, csv_path=None):
    print("Stage 1: loading data...")
    df = load_from_csv(csv_path) if csv_path else load_dataset_auto(prefer_full=use_full)
    print(f"  -> {len(df)} claims loaded.")
    has_gold = "gold_label" in df.columns

    ml_preds, ml_probas, ml_test_uids = {}, {}, set()
    if has_gold:
        from nlp_classifier import MLClassifier
        from sklearn.model_selection import train_test_split
        idx_train, idx_test = train_test_split(df.index, test_size=0.3, random_state=42,
            stratify=df["gold_label"] if df["gold_label"].nunique()>1 else None)
        ml_test_uids = set(df.loc[idx_test, "unique_id"])
        ml = MLClassifier().fit(df.loc[idx_train,"claim"].tolist(), df.loc[idx_train,"gold_label"].tolist())
        preds = ml.predict(df["claim"].tolist())
        probas = ml.predict_proba_greenwashing(df["claim"].tolist())
        for i, uid in enumerate(df["unique_id"]):
            ml_preds[uid] = preds[i]; ml_probas[uid] = round(float(probas[i]),3)
        print(f"  -> trained on {len(idx_train)} rows, held out {len(idx_test)} for testing")
        print("  -> top greenwashing terms:", ml.top_features(8))

    results = []
    for _, row in df.iterrows():
        uid = row["unique_id"]; claim = row["claim"]; company = row.get("company",""); gold = row.get("gold_label") if has_gold else None
        meta = extract_metadata(claim)
        v = build_verdict(claim, company, gold_label=gold, ml_label=ml_preds.get(uid), ml_probability=ml_probas.get(uid))
        results.append({"unique_id": int(uid), "claim": v.claim, "company": company, "industry": v.industry,
            "final_label": v.final_label, "confidence": v.confidence, "claim_type": v.claim_type,
            "specificity_score": v.specificity_score, "greenwashing_type": v.greenwashing_type,
            "reasons": v.reasons, "highlighted_terms": v.highlighted_terms, "gold_label": gold,
            "ml_label": v.ml_label, "ml_probability": v.ml_probability, "word_count": meta["word_count"],
            "is_ml_test_row": uid in ml_test_uids})

    with open("processed_claims.json","w") as f: json.dump(results, f, indent=2)
    summary = {"total_claims": len(results),
        "greenwashing_flagged": sum(1 for r in results if r["final_label"]=="greenwashing"),
        "by_company": dict(Counter(r["company"] for r in results)),
        "by_type": dict(Counter(r["greenwashing_type"] for r in results)),
        "by_industry": dict(Counter(r["industry"] for r in results))}
    with open("dashboard_summary.json","w") as f: json.dump(summary, f, indent=2)

    if has_gold:
        metrics = compute_metrics(results)
        with open("metrics_report.json","w") as f: json.dump(metrics, f, indent=2)
        print("\n--- METRICS ---"); print(json.dumps(metrics, indent=2))
    print("\nDone.")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--full", action="store_true")
    parser.add_argument("--csv", type=str, default=None)
    args = parser.parse_args()
    main(use_full=args.full, csv_path=args.csv)

Writing run_pipeline.py


In [8]:
!python run_pipeline.py --full

Stage 1: loading data...
README.md: 100% 4.54k/4.54k [00:00<00:00, 6.63MB/s]
emerald_data.csv: 100% 204k/204k [00:00<00:00, 10.2MB/s]
Generating train split: 100% 620/620 [00:00<00:00, 24256.52 examples/s]
  -> 620 claims loaded.
  -> trained on 434 rows, held out 186 for testing
  -> top greenwashing terms: [('million', 1.111), ('power', 0.889), ('electric', 0.762), ('2021', 0.757), ('program', 0.734), ('aims', 0.687), ('2040', 0.657), ('plans', 0.646)]

--- METRICS ---
{
  "n_evaluated": 620,
  "rule_based (all rows)": {
    "accuracy": 0.694,
    "precision": 0.586,
    "recall": 0.529,
    "f1": 0.556,
    "tp": 119,
    "fp": 84,
    "fn": 106,
    "tn": 311
  },
  "ml_model (held-out test split only)": {
    "accuracy": 0.742,
    "precision": 0.667,
    "recall": 0.588,
    "f1": 0.625,
    "tp": 40,
    "fp": 20,
    "fn": 28,
    "tn": 98
  },
  "ml_test_set_size": 186
}

Done.


In [9]:
from google.colab import files
uploaded = files.upload()   # choose dashboard_template.html

Saving GreenLens — Greenwashing Claim Auditor.html to GreenLens — Greenwashing Claim Auditor.html


# **DASHBOARD**

In [11]:
import json
claims = json.load(open("processed_claims.json"))
summary = json.load(open("dashboard_summary.json"))
metrics = json.load(open("metrics_report.json"))
tpl = open("GreenLens — Greenwashing Claim Auditor.html", encoding="utf-8").read()
tpl = tpl.replace("/*__CLAIMS__*/[]/*__END_CLAIMS__*/", json.dumps(claims))
tpl = tpl.replace("/*__SUMMARY__*/{}/*__END_SUMMARY__*/", json.dumps(summary))
tpl = tpl.replace("/*__METRICS__*/{}/*__END_METRICS__*/", json.dumps(metrics))
open("dashboard.html", "w", encoding="utf-8").write(tpl)
print("done")

done


In [13]:
from google.colab import files
for f in ["dashboard.html", "processed_claims.json", "metrics_report.json"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>